# Kazakh Legal RAG — Generator Comparison (standalone)

This notebook does NOT repeat retrieval ablations. It loads the corpus + benchmark, builds **only** the Static Hybrid retriever (BM25 + BGE-M3) needed to construct RAG evidence, then runs the generator backbone comparison and a **Claude** LLM-judge.

Run order: 0.x setup -> 1 build retriever -> 2 generate -> 3 judge (Claude) -> 4 Table 22.

In [ ]:
#@title 0.1 Install dependencies
!pip install -q datasets rank-bm25 sentence-transformers faiss-cpu pandas numpy matplotlib openpyxl tqdm scikit-learn scipy openai

In [ ]:
#@title 0.2 Imports and global configuration
import os, re, json, time, shutil, zipfile, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
warnings.filterwarnings('ignore')

BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
HF_CORPUS_ID  = 'Arailym-tleubayeva/LegalRAG'
OUTPUT_DIR    = Path('/content/legalrag_outputs')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 256
RRF_K = 60
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Output dir: {OUTPUT_DIR}')


def norm_id(v):
    """Normalize ids read from CSV/HF datasets, especially float-like ids such as 123.0."""
    if pd.isna(v):
        return ''
    s = str(v).strip()
    if s.lower() in {'', 'nan', 'none', 'null'}:
        return ''
    return re.sub(r'\.0$', '', s)


def clean_str(v):
    if pd.isna(v):
        return ''
    return str(v).strip()


def save_table(df: pd.DataFrame, name: str):
    """Save a result table as CSV and XLSX under OUTPUT_DIR."""
    csv_path = OUTPUT_DIR / f'{name}.csv'
    xlsx_path = OUTPUT_DIR / f'{name}.xlsx'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    df.to_excel(xlsx_path, index=False)
    print(f'Saved: {csv_path.name}, {xlsx_path.name}')


def save_json(obj, name: str):
    path = OUTPUT_DIR / f'{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'Saved: {path.name}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Output dir: /content/legalrag_outputs


In [ ]:
#@title 0.3 clean_str with float/ID normalization (FIX for 788.0 vs "788" bug)
import math

def clean_str(x):
    """Normalize any value to a clean string ID.
    Critical fix: pandas reads numeric chunk IDs as float (788.0), while corpus
    chunk IDs are strings ('788'). Without normalization, gold==retrieved checks
    silently fail. This maps 788.0 / '788.0' / 788 / '788' all to '788'."""
    if x is None:
        return ''
    if isinstance(x, float):
        if math.isnan(x):
            return ''
        if x.is_integer():
            return str(int(x))
        return str(x).strip()
    if isinstance(x, int):
        return str(x)
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return ''
    if s.endswith('.0') and s[:-2].isdigit():
        s = s[:-2]
    return s

assert clean_str(788.0) == clean_str('788') == clean_str(788) == clean_str('788.0') == '788', 'clean_str fix failed'
print('clean_str fixed: 788.0 / "788.0" / 788 / "788" all ->', clean_str(788.0))

clean_str fixed: 788.0 / "788.0" / 788 / "788" all -> 788


In [ ]:
#@title 1.1 Load `legalrag_qa_benchmark.csv`
from pathlib import Path

# Colab upload fallback
if not Path(BENCHMARK_CSV).exists():
    local_fallbacks = [
        '/mnt/data/legalrag_qa_benchmark(2).csv',
        '/mnt/data/legalrag_qa_benchmark.csv',
        'legalrag_qa_benchmark.csv',
    ]
    found = next((p for p in local_fallbacks if Path(p).exists()), None)
    if found:
        BENCHMARK_CSV = found
        print(f'Using local fallback: {BENCHMARK_CSV}')
    else:
        try:
            from google.colab import files as cf
            uploaded = cf.upload()
            for fname in uploaded:
                Path(fname).rename('/content/legalrag_qa_benchmark.csv')
            BENCHMARK_CSV = '/content/legalrag_qa_benchmark.csv'
        except Exception as e:
            raise FileNotFoundError('Benchmark CSV not found. Upload legalrag_qa_benchmark.csv.') from e

bench_all = pd.read_csv(BENCHMARK_CSV, encoding='utf-8-sig')
bench_all = bench_all.reset_index(drop=False).rename(columns={'index': 'row_id'})

required_cols = [
    'question_id', 'question', 'gold_answer', 'gold_evidence', 'gold_chunk_id',
    'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph', 'language',
    'answerability', 'legal_domain', 'complexity', 'question_type'
]
missing = [c for c in required_cols if c not in bench_all.columns]
if missing:
    raise ValueError(f'Missing benchmark columns: {missing}')

for col in ['gold_chunk_id', 'gold_doc_id', 'gold_source', 'gold_article', 'gold_paragraph']:
    bench_all[f'{col}_str'] = bench_all[col].apply(norm_id if 'id' in col else clean_str)

bench_all['answerability'] = bench_all['answerability'].fillna('').astype(str).str.strip()
bench_all['complexity'] = bench_all['complexity'].fillna('').astype(str).str.strip()
bench_all['question_type'] = bench_all['question_type'].fillna('').astype(str).str.strip()
bench_all['legal_domain'] = bench_all['legal_domain'].fillna('').astype(str).str.strip()
bench_all['question'] = bench_all['question'].fillna('').astype(str)

bench_ans = bench_all[bench_all['answerability'].eq('answerable')].reset_index(drop=True)
bench_unans = bench_all[bench_all['answerability'].eq('unanswerable')].reset_index(drop=True)

print(f'Benchmark rows: {len(bench_all)}')
print(f'Answerable: {len(bench_ans)} | Unanswerable: {len(bench_unans)}')
print(f'Unique questions: {bench_all["question"].nunique()}')
print('Answerable complexity:', dict(bench_ans['complexity'].value_counts()))

Benchmark rows: 242
Answerable: 182 | Unanswerable: 60
Unique questions: 186
Answerable complexity: {'complex': np.int64(75), 'moderate': np.int64(65), 'simple': np.int64(42)}


In [ ]:
#@title 0.4 Normalize gold ID columns at load time (root-cause fix)
ID_COLS = ['gold_chunk_id', 'gold_doc_id', 'gold_chunk_id_str', 'gold_doc_id_str',
           'gold_source', 'gold_source_str']

def _normalize_id_columns(df):
    for col in ID_COLS:
        if col in df.columns:
            df[col] = (df[col].astype('string')
                       .str.replace(r'\.0$', '', regex=True)
                       .fillna(''))
    return df

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        globals()[_name] = _normalize_id_columns(globals()[_name].copy())
        print(f'Normalized ID columns in {_name} ({len(globals()[_name])} rows)')

for _name in ['bench_all', 'bench_ans', 'bench_unans']:
    if _name in dir():
        _df = globals()[_name]
        if 'gold_chunk_id_str' not in _df.columns and 'gold_chunk_id' in _df.columns:
            _df['gold_chunk_id_str'] = _df['gold_chunk_id'].map(clean_str)
        if 'gold_doc_id_str' not in _df.columns and 'gold_doc_id' in _df.columns:
            _df['gold_doc_id_str'] = _df['gold_doc_id'].map(clean_str)
        if 'gold_source_str' not in _df.columns and 'gold_source' in _df.columns:
            _df['gold_source_str'] = _df['gold_source'].map(clean_str)

print('Gold ID columns normalized and *_str helpers ensured.')

Normalized ID columns in bench_all (242 rows)
Normalized ID columns in bench_ans (182 rows)
Normalized ID columns in bench_unans (60 rows)
Gold ID columns normalized and *_str helpers ensured.


In [ ]:
#@title 2.1 Load `Arailym-tleubayeva/LegalRAG`
from datasets import load_dataset

ds = load_dataset(HF_CORPUS_ID)
split = 'train' if 'train' in ds else list(ds.keys())[0]
corpus_df = ds[split].to_pandas().reset_index(drop=True)

print(f'HF dataset: {HF_CORPUS_ID} | split={split}')
print(f'Corpus rows: {len(corpus_df):,}')
print('Columns:', corpus_df.columns.tolist())


def choose_col(candidates, required=True):
    for c in candidates:
        if c in corpus_df.columns:
            return c
    if required:
        raise ValueError(f'None of these corpus columns found: {candidates}')
    return None

CHUNK_COL = choose_col(['chunk_id', 'id', 'chunkId'])
DOC_COL = choose_col(['doc_id', 'document_id', 'docid', 'documentId'], required=False)
SOURCE_COL = choose_col(['source', 'url', 'source_url', 'link'], required=False)
TEXT_COL = choose_col(['text', 'content', 'chunk_text'])
NORM_TEXT_COL = choose_col(['normalized_text', 'text_normalized', 'norm_text'], required=False)
TITLE_COL = choose_col(['title', 'doc_title', 'document_title'], required=False)

corpus_df['chunk_id_str'] = corpus_df[CHUNK_COL].apply(norm_id)
corpus_df['doc_id_str'] = corpus_df[DOC_COL].apply(norm_id) if DOC_COL else ''
corpus_df['source_str'] = corpus_df[SOURCE_COL].apply(clean_str) if SOURCE_COL else ''
corpus_df['text_str'] = corpus_df[TEXT_COL].fillna('').astype(str)
corpus_df['title_str'] = corpus_df[TITLE_COL].fillna('').astype(str) if TITLE_COL else ''

c_ids = corpus_df['chunk_id_str'].tolist()
c_docs = corpus_df['doc_id_str'].tolist()
c_srcs = corpus_df['source_str'].tolist()
c_text = corpus_df['text_str'].tolist()
c_titles = corpus_df['title_str'].tolist()

id_to_pos = {cid: i for i, cid in enumerate(c_ids)}
id2text = dict(zip(c_ids, c_text))
id2doc = dict(zip(c_ids, c_docs))
id2src = dict(zip(c_ids, c_srcs))
id2title = dict(zip(c_ids, c_titles))

gold_chunks = set(bench_ans['gold_chunk_id_str'].dropna().astype(str)) - {''}
covered = gold_chunks & set(c_ids)
coverage = len(covered) / max(1, len(gold_chunks))
print(f'Gold chunk coverage: {len(covered)}/{len(gold_chunks)} ({coverage:.1%})')

corpus_stats = pd.DataFrame([
    ('HF corpus id', HF_CORPUS_ID),
    ('Split', split),
    ('Rows / chunks', len(corpus_df)),
    ('Chunk id column', CHUNK_COL),
    ('Document id column', DOC_COL or ''),
    ('Source column', SOURCE_COL or ''),
    ('Text column', TEXT_COL),
    ('Normalized text column', NORM_TEXT_COL or 'not available'),
    ('Gold chunk coverage', f'{len(covered)}/{len(gold_chunks)} ({coverage:.1%})'),
], columns=['Statistic', 'Value'])
display(corpus_stats)
save_table(corpus_stats, 'table00_corpus_loading_stats')

README.md:   0%|          | 0.00/5.74k [00:00<?, ?B/s]

chunked_all_docs_structured_fixed.json:   0%|          | 0.00/256M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/56916 [00:00<?, ? examples/s]

HF dataset: Arailym-tleubayeva/LegalRAG | split=train
Corpus rows: 56,916
Columns: ['doc_id', 'date', 'section_title', 'source', 'chunk_id', 'text', 'words', 'char_len', 'word_len', 'normalized_text']
Gold chunk coverage: 182/182 (100.0%)


,Statistic,Value
0,HF corpus id,Arailym-tleubayeva/LegalRAG
1,Split,train
2,Rows / chunks,56916
3,Chunk id column,chunk_id
4,Document id column,doc_id
5,Source column,source
6,Text column,text
7,Normalized text column,normalized_text
8,Gold chunk coverage,182/182 (100.0%)


Saved: table00_corpus_loading_stats.csv, table00_corpus_loading_stats.xlsx


In [ ]:
#@title 3.1 Retrieval result object and metrics

def make_result(indices, scores=None, score_name='score'):
    ids, docs, srcs, texts, titles, scs = [], [], [], [], [], []
    if scores is None:
        scores = [np.nan] * len(indices)
    for idx, score in zip(indices, scores):
        idx = int(idx)
        if idx < 0 or idx >= len(c_ids):
            continue
        ids.append(c_ids[idx])
        docs.append(c_docs[idx])
        srcs.append(c_srcs[idx])
        texts.append(c_text[idx])
        titles.append(c_titles[idx])
        scs.append(float(score) if score is not None and not pd.isna(score) else np.nan)
    return {
        'retrieved_ids': ids,
        'retrieved_docs': docs,
        'retrieved_sources': srcs,
        'retrieved_texts': texts,
        'retrieved_titles': titles,
        'retrieved_scores': scs,
        'score_name': score_name,
    }


def hit_at_1(ids, gold_set):
    return int(bool(ids) and ids[0] in gold_set)


def recall_at_k(ids, gold_set, k):
    if not gold_set:
        return np.nan
    return len(set(ids[:k]) & gold_set) / len(gold_set)


def mrr_at_k(ids, gold_set, k=10):
    for rank, cid in enumerate(ids[:k], 1):
        if cid in gold_set:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ids, gold_set, k=10):
    if not gold_set:
        return np.nan
    dcg = sum(1.0 / np.log2(rank + 1) for rank, cid in enumerate(ids[:k], 1) if cid in gold_set)
    ideal_hits = min(len(gold_set), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg else 0.0


def exact_value_recall_at_k(values, gold_value, k):
    gold_value = clean_str(gold_value)
    if not gold_value:
        return np.nan
    return int(gold_value in [clean_str(v) for v in values[:k]])


def eval_retrieval(results, bench, kset=(1, 3, 5, 10)):
    """Evaluate retrieval on answerable questions with gold_chunk_id/gold_doc_id/gold_source."""
    rows = []
    for res, (_, row) in zip(results, bench.iterrows()):
        # FIX: support one OR several gold chunk ids separated by | ; ,
        gold_chunk = {p for p in re.split(r'[|;,]\s*', clean_str(row.get('gold_chunk_id_str', ''))) if p} - {''}
        ids = [clean_str(x) for x in res.get('retrieved_ids', [])]
        docs = [clean_str(x) for x in res.get('retrieved_docs', [])]
        srcs = [clean_str(x) for x in res.get('retrieved_sources', [])]
        scores = res.get('retrieved_scores', [])
        out = {
            'question_id': row.get('question_id', ''),
            'row_id': row.get('row_id', ''),
            'complexity': row.get('complexity', ''),
            'question_type': row.get('question_type', ''),
            'legal_domain': row.get('legal_domain', ''),
            'Hit@1': hit_at_1(ids, gold_chunk),
            'MRR@10': mrr_at_k(ids, gold_chunk, 10),
            'nDCG@10': ndcg_at_k(ids, gold_chunk, 10),
            'DocRecall@10': exact_value_recall_at_k(docs, row.get('gold_doc_id_str', ''), 10),
            'SourceRecall@10': exact_value_recall_at_k(srcs, row.get('gold_source_str', ''), 10),
            'TopScore': float(scores[0]) if scores else np.nan,
            'TopChunk': ids[0] if ids else '',
            'GoldChunk': next(iter(gold_chunk)) if gold_chunk else '',
        }
        for k in kset:
            out[f'Recall@{k}'] = recall_at_k(ids, gold_chunk, k)
        rows.append(out)
    details = pd.DataFrame(rows)
    metric_cols = [
        'Hit@1', 'Recall@1', 'Recall@3', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10',
        'DocRecall@10', 'SourceRecall@10', 'TopScore'
    ]
    agg = {c: round(float(np.nanmean(details[c])), 4) for c in metric_cols if c in details.columns}
    return agg, details


def summarize_metric_table(rows, name, sort_col='Recall@10'):
    df = pd.DataFrame(rows)
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    display(df)
    save_table(df, name)
    return df

print('Metrics ready.')

Metrics ready.


In [ ]:
#@title 3.2 Tokenization, BM25, dense retrieval, and RRF helpers
from rank_bm25 import BM25Okapi


def tok(text):
    return str(text).lower().split()


def build_bm25(texts, tokenizer=tok, desc='BM25 corpus'):
    tokenized = [tokenizer(t) for t in tqdm(texts, desc=desc)]
    return BM25Okapi(tokenized)


def bm25_ret(index, query, k=10, tokenizer=tok):
    scores = index.get_scores(tokenizer(query))
    top = np.argsort(scores)[::-1][:k]
    return make_result(top, scores[top], score_name='bm25')


def dense_ret(index, query_embedding, k=10):
    scores, idx = index.search(query_embedding.reshape(1, -1).astype('float32'), k)
    return make_result(idx[0], scores[0], score_name='dense_ip')


def rrf(result_list, k=10, rrf_k=RRF_K):
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids = res.get('retrieved_ids', [])
        for rank, cid in enumerate(ids, 1):
            scores[cid] += 1.0 / (rrf_k + rank)
            pos = ids.index(cid)
            meta[cid] = {
                'doc': res.get('retrieved_docs', [''])[pos] if pos < len(res.get('retrieved_docs', [])) else '',
                'source': res.get('retrieved_sources', [''])[pos] if pos < len(res.get('retrieved_sources', [])) else '',
                'text': res.get('retrieved_texts', [''])[pos] if pos < len(res.get('retrieved_texts', [])) else '',
                'title': res.get('retrieved_titles', [''])[pos] if pos < len(res.get('retrieved_titles', [])) else '',
            }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids': top_ids,
        'retrieved_docs': [meta[cid]['doc'] for cid in top_ids],
        'retrieved_sources': [meta[cid]['source'] for cid in top_ids],
        'retrieved_texts': [meta[cid]['text'] for cid in top_ids],
        'retrieved_titles': [meta[cid]['title'] for cid in top_ids],
        'retrieved_scores': [scores[cid] for cid in top_ids],
        'score_name': 'rrf',
    }


def retrieval_confidence(res):
    scores = res.get('retrieved_scores', [])
    return float(scores[0]) if scores else 0.0

print('Retrieval utilities ready.')

Retrieval utilities ready.


In [ ]:
#@title 4.1 Build original BM25 index
print('Building BM25 on original corpus text...')
bm25_text = build_bm25(c_text, tok, desc='BM25-original')
print('BM25 ready.')

Building BM25 on original corpus text...


BM25-original:   0%|          | 0/56916 [00:00<?, ?it/s]

BM25 ready.


In [ ]:
#@title 1.0 rrf FIX — rank-aligned text/doc (overrides the buggy ids.index version)
def rrf(result_list, k=10, rrf_k=RRF_K):
    """Reciprocal-rank fusion. FIX: metadata taken at the current rank position,
    not via ids.index(cid), so retrieved_texts/docs stay aligned with retrieved_ids
    even when an id appears in both BM25 and dense lists."""
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids    = res.get('retrieved_ids', [])
        docs   = res.get('retrieved_docs', [])
        srcs   = res.get('retrieved_sources', [])
        texts  = res.get('retrieved_texts', [])
        titles = res.get('retrieved_titles', [])
        for rank, cid in enumerate(ids):
            scores[cid] += 1.0 / (rrf_k + rank + 1)
            if cid not in meta:
                meta[cid] = {
                    'doc':    docs[rank]   if rank < len(docs)   else '',
                    'source': srcs[rank]   if rank < len(srcs)   else '',
                    'text':   texts[rank]  if rank < len(texts)  else '',
                    'title':  titles[rank] if rank < len(titles) else '',
                }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids':     top_ids,
        'retrieved_docs':    [meta[c]['doc']    for c in top_ids],
        'retrieved_sources': [meta[c]['source'] for c in top_ids],
        'retrieved_texts':   [meta[c]['text']   for c in top_ids],
        'retrieved_titles':  [meta[c]['title']  for c in top_ids],
        'retrieved_scores':  [scores[c]         for c in top_ids],
        'score_name': 'rrf',
    }

# verify alignment
_q = bench_ans.iloc[0]['question']
_c = rrf([bm25_ret(bm25_text, _q, 20), dense_ret(faiss_index, query_embs_ans[0], 20)], k=20) if 'faiss_index' in dir() else None
print('rrf overridden (rank-aligned).')


rrf overridden (rank-aligned).


In [ ]:
#@title 1.1 Build BGE-M3 dense index (only model needed for Static Hybrid)
import faiss
from sentence_transformers import SentenceTransformer

DENSE_MODEL_ID = 'BAAI/bge-m3'
MAX_SEQ_LENGTH = 512
if torch.cuda.is_available():
    _vram = torch.cuda.get_device_properties(0).total_memory/1e9
    DENSE_BATCH_SIZE = 128 if _vram >= 40 else (32 if _vram >= 24 else (16 if _vram >= 12 else 4))
else:
    DENSE_BATCH_SIZE = 32
print(f'Dense batch size: {DENSE_BATCH_SIZE}, max_seq_len: {MAX_SEQ_LENGTH}')

dense_model = SentenceTransformer(DENSE_MODEL_ID, device=DEVICE)
dense_model.max_seq_length = MAX_SEQ_LENGTH

print('Encoding corpus...')
corpus_emb = dense_model.encode(c_text, batch_size=DENSE_BATCH_SIZE, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True).astype('float32')
faiss_index = faiss.IndexFlatIP(corpus_emb.shape[1])
faiss_index.add(corpus_emb)

print('Encoding answerable questions...')
query_embs_ans = dense_model.encode(bench_ans['question'].tolist(), batch_size=DENSE_BATCH_SIZE,
                                     show_progress_bar=True, convert_to_numpy=True,
                                     normalize_embeddings=True).astype('float32')
print('Dense index ready:', faiss_index.ntotal, 'vectors')


Dense batch size: 128, max_seq_len: 512


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Encoding corpus...


Batches:   0%|          | 0/445 [00:00<?, ?it/s]

Encoding answerable questions...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Dense index ready: 56916 vectors


In [ ]:
#@title 1.2 Static Hybrid retrieval (BM25+BGE-M3) for 182 answerable
STATIC_K = 10

def static_hybrid_retrieve(question, query_embedding, k=STATIC_K):
    return rrf([bm25_ret(bm25_text, question, k=k),
               dense_ret(faiss_index, query_embedding, k=k)], k=k)

static_res = [static_hybrid_retrieve(row['question'], query_embs_ans[i])
              for i, row in tqdm(list(bench_ans.iterrows()), desc='Static Hybrid (182)')]

# alignment check (must be True after rrf fix)
_g = clean_str(bench_ans.iloc[0]['gold_chunk_id_str'])
_ids = [clean_str(x) for x in static_res[0]['retrieved_ids']]
if _g in _ids:
    _p = _ids.index(_g)
    print('RRF text == id2text ?', static_res[0]['retrieved_texts'][_p][:80] == id2text.get(static_res[0]['retrieved_ids'][_p],'')[:80])
print('Static retrieval ready for', len(static_res), 'questions.')

def evidence_context_from_result(res, max_chunks=10, max_chars=1200):
    blocks = []
    for rank, (cid, doc, src, title, text) in enumerate(zip(
        res.get('retrieved_ids', []), res.get('retrieved_docs', []),
        res.get('retrieved_sources', []), res.get('retrieved_titles', []),
        res.get('retrieved_texts', [])), 1):
        if rank > max_chunks: break
        t = str(text).replace('\n',' ').strip()[:max_chars]
        blocks.append(f'[E{rank}] chunk_id={cid}; doc={doc}; src={src}\n{t}')
    return '\n\n'.join(blocks)


Static Hybrid (182):   0%|          | 0/182 [00:00<?, ?it/s]

RRF text == id2text ? False
Static retrieval ready for 182 questions.


---
## 2. Generator backbone comparison

In [ ]:
#@title 2.1 Generator comparison — configuration
import gc
RUN_GENERATION = False        # set True to generate (GPU + API budget)
GEN_EVAL_N = 50               # questions per system (stratified)
GEN_MAX_NEW_TOKENS = 512

# (display_name, model_id, backend, setting, language)
GEN_SYSTEMS = [
    ('KazLLM-1.0-8B',         'issai/LLama-3.1-KazLLM-1.0-8B',       'hf',     'LLM-only', 'kz'),
    ('Hermes-3-Llama-3.1-8B', 'NousResearch/Hermes-3-Llama-3.1-8B',  'hf',     'LLM-only', 'kz'),
    ('Qwen2.5-7B',            'Qwen/Qwen2.5-7B-Instruct',            'hf',     'LLM-only', 'kz'),
    ('GPT-4.1 (LLM-only)',    'gpt-4.1',                             'openai', 'LLM-only', 'kz'),
    ('GPT-4.1 (RAG KZ)',      'gpt-4.1',                             'openai', 'RAG',      'kz'),
    ('GPT-4.1 (RAG RUS)',     'gpt-4.1',                             'openai', 'RAG',      'rus'),
]

_pool = bench_ans.copy()
if 'complexity' in _pool.columns and len(_pool) > GEN_EVAL_N:
    eval_subset = (_pool.groupby('complexity', group_keys=False)
                   .apply(lambda s: s.sample(min(len(s), max(1, GEN_EVAL_N//3)), random_state=RANDOM_SEED))
                   .head(GEN_EVAL_N).reset_index(drop=True))
else:
    eval_subset = _pool.head(GEN_EVAL_N).reset_index(drop=True)
_rowpos = {rid: i for i, rid in enumerate(bench_ans['row_id'].tolist())}
def evidence_for_row(row):
    p = _rowpos.get(row['row_id'])
    return evidence_context_from_result(static_res[p], max_chunks=10) if p is not None else ''
print('Eval subset:', len(eval_subset), 'questions')

PROMPTS = {
 'kz': {'LLM-only':"Қазақ тіліндегі заң сұрағына нақты жауап беріңіз. Сенімді болмасаңыз, анықтау мүмкін емес деп жазыңыз.\n\nСұрақ:\n{q}",
        'RAG':"Берілген дереккөздерді пайдаланып қазақ тіліндегі заң сұрағына жауап беріңіз. Дереккөз жеткіліксіз болса бас тартыңыз. [E1],[E2] сілтеме жасаңыз.\n\nСұрақ:\n{q}\n\nДереккөздер:\n{e}"},
 'rus':{'LLM-only':"Дайте точный ответ на юридический вопрос. Если не уверены, укажите что ответ не может быть определён.\n\nВопрос:\n{q}",
        'RAG':"Используя источники, ответьте на юридический вопрос. Если недостаточно — откажитесь. Цитируйте [E1],[E2].\n\nВопрос:\n{q}\n\nИсточники:\n{e}"},
}
def build_prompt(setting, lang, q, e=''):
    return PROMPTS[lang][setting].format(q=q, e=e)


Eval subset: 242 questions (182 answerable + 60 unanswerable)
Building Static Hybrid retrieval for all 242 (needed for unanswerable evidence)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Static retrieval (242):   0%|          | 0/242 [00:00<?, ?it/s]

In [ ]:
#@title 2.2 Backends — local HF (4-bit) + OpenAI generator
_HF = {}
def load_hf_4bit(mid):
    if mid in _HF: return _HF[mid]
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map='auto',
                                                 trust_remote_code=True, dtype=torch.float16).eval()
    _HF[mid] = (tok, model); return tok, model
def free_hf(mid):
    if mid in _HF: del _HF[mid]
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
def gen_hf(mid, prompt, max_new=GEN_MAX_NEW_TOKENS):
    tok, model = load_hf_4bit(mid)
    try:
        text = tok.apply_chat_template([{'role':'user','content':prompt}], add_generation_prompt=True, tokenize=False)
    except Exception:
        text = prompt
    enc = tok(text, return_tensors='pt', truncation=True, max_length=4096)
    enc = {k: v.to(model.device) for k,v in enc.items()}
    ilen = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ilen:], skip_special_tokens=True).strip()
def gen_openai(mid, prompt, max_new=GEN_MAX_NEW_TOKENS):
    try:
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=mid, temperature=0, max_tokens=max_new,
                                             messages=[{'role':'user','content':prompt}])
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'GEN_FAILED: {type(e).__name__}: {e}'


In [ ]:
#@title 2.3 Run generation across systems
if RUN_GENERATION:
    recs = []
    for name, mid, backend, setting, lang in GEN_SYSTEMS:
        print(f'\n=== {name} | {backend} | {setting} | {lang} ===')
        for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc=name):
            ev = evidence_for_row(row) if setting=='RAG' else ''
            prompt = build_prompt(setting, lang, row['question'], ev)
            try:
                ans = gen_hf(mid, prompt) if backend=='hf' else gen_openai(mid, prompt)
            except Exception as e:
                ans = f'GEN_FAILED: {type(e).__name__}: {e}'
            recs.append({'system':name,'model_id':mid,'backend':backend,'setting':setting,'language':lang,
                         'row_id':row['row_id'],'question':row['question'],
                         'gold_answer':row.get('gold_answer',''),'gold_chunk_id':row.get('gold_chunk_id_str',''),
                         'generated_answer':ans})
        if backend=='hf': free_hf(mid)
    gen_compare_df = pd.DataFrame(recs)
    save_table(gen_compare_df, 'gen_compare_raw')
    display(gen_compare_df.groupby(['system','setting']).size().rename('n').reset_index())
else:
    print('RUN_GENERATION is False.'); gen_compare_df = pd.DataFrame()



=== KazLLM-1.0-8B | hf | LLM-only | kz ===


KazLLM-1.0-8B:   0%|          | 0/242 [00:00<?, ?it/s]


=== Hermes-3-Llama-3.1-8B | hf | LLM-only | kz ===


Hermes-3-Llama-3.1-8B:   0%|          | 0/242 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

...

GPT-4.1 (RAG KZ):   0%|          | 0/242 [00:00<?, ?it/s]


=== GPT-4.1 (RAG RUS) | openai | RAG | rus ===


GPT-4.1 (RAG RUS):   0%|          | 0/242 [00:00<?, ?it/s]

Saved: gen_compare_raw.csv, gen_compare_raw.xlsx


,system,setting,n
0,GPT-4.1 (LLM-only),LLM-only,242
1,GPT-4.1 (RAG KZ),RAG,242
2,GPT-4.1 (RAG RUS),RAG,242
3,Hermes-3-Llama-3.1-8B,LLM-only,242
4,KazLLM-1.0-8B,LLM-only,242
5,Qwen2.5-7B,LLM-only,242


---
## 3. Claude LLM-judge + Table 22

In [ ]:
#@title 3.1 LLM-judge (GPT-4o, primary; matches FINAL_v3 cell 10.3f)
RUN_JUDGE = False
JUDGE_MODEL = 'gpt-4o'  # primary judge; pin exact snapshot + access date in the paper. GPT-5 is the secondary cross-judge (FINAL_v3 cell 10.3h).
import re as _re, json as _json

JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER to the GOLD ANSWER for the QUESTION.
Score each dimension as an integer 0-2 (hallucination: 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}"""

def judge_one(question, gold, answer):
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=200,
              messages=[{'role':'user','content':JUDGE_PROMPT.format(question=question, gold=gold, answer=answer)}])
        txt = resp.choices[0].message.content
        txt = _re.sub(r'```json|```','',txt).strip()
        return _json.loads(txt)
    except Exception as e:
        return {'_error': f'{type(e).__name__}: {e}'}

if RUN_JUDGE and len(gen_compare_df):
    rows = []
    for _, r in tqdm(gen_compare_df.iterrows(), total=len(gen_compare_df), desc='Claude-judge'):
        s = judge_one(r['question'], r['gold_answer'], r['generated_answer'])
        rec = {**r.to_dict()}
        for d in ['legal_correctness','evidence_support','completeness','clarity','hallucination']:
            rec[f'judge_{d}'] = s.get(d, np.nan) if '_error' not in s else np.nan
        rows.append(rec)
    judged_df = pd.DataFrame(rows)
    save_table(judged_df, 'gen_compare_judged_claude')
    print('Judged', len(judged_df), 'answers with Claude.')
else:
    print('RUN_JUDGE is False or no generations.')


Claude-judge:   0%|          | 0/1452 [00:00<?, ?it/s]

Saved: gen_compare_judged_claude.csv, gen_compare_judged_claude.xlsx
Judged 1452 answers with Claude.


In [ ]:
#@title 3.2 Table 22 — Generator backbone comparison (Claude judge)
if 'judged_df' in dir() and len(judged_df):
    import numpy as np
    df = judged_df.copy()
    ABST = ['не могу','недостаточно','жоқ','мүмкін емес','нет информации','cannot','insufficient']
    df['abstain'] = df['generated_answer'].astype(str).str.lower().apply(lambda s: int(any(m in s for m in ABST)))
    df['answerability'] = df['row_id'].map(dict(zip(bench_all['row_id'], bench_all['answerability'])))
    LABEL = {('GPT-4.1 (RAG KZ)','RAG'):('GPT-4.1','Static Hybrid RAG KZ'),
             ('GPT-4.1 (RAG RUS)','RAG'):('GPT-4.1','Static Hybrid RAG RUS'),
             ('GPT-4.1 (LLM-only)','LLM-only'):('GPT-4.1','LLM-only')}
    rows=[]
    for (sysn,setting), sub in df.groupby(['system','setting']):
        lab = LABEL.get((sysn,setting),(sysn,setting))
        ans_sub = sub[sub['answerability']=='answerable']; un_sub = sub[sub['answerability']=='unanswerable']
        rows.append({'System':lab[0],'Setting':lab[1],
            'Legal':round(float(sub['judge_legal_correctness'].mean()),3),
            'Evidence':round(float(sub['judge_evidence_support'].mean()),3),
            'Halluc_down':round(float(sub['judge_hallucination'].mean()),3),
            'Completeness':round(float(sub['judge_completeness'].mean()),3),
            'Clarity':round(float(sub['judge_clarity'].mean()),3),
            'CorrectAbstain_up':round(float(un_sub['abstain'].mean()),3) if len(un_sub) else None,
            'FalseAbstain_down':round(float(ans_sub['abstain'].mean()),3) if len(ans_sub) else None})
    table22 = pd.DataFrame(rows)
    display(table22); save_table(table22,'table22_generator_backbone_comparison')
    print('Judge = GPT-4o (primary). NOTE: shares family with GPT-4.1 -> report self-preference bias as a limitation; GPT-5 is the secondary cross-judge.')
else:
    print('Need judged_df first.')


,System,Setting,Legal,Evidence,Halluc_down,Completeness,Clarity,CorrectAbstain_up,FalseAbstain_down
0,GPT-4.1,LLM-only,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,GPT-4.1,Static Hybrid RAG KZ,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,GPT-4.1,Static Hybrid RAG RUS,NaN,NaN,NaN,NaN,NaN,0.0,0.0
3,Hermes-3-Llama-3.1-8B,LLM-only,NaN,NaN,NaN,NaN,NaN,0.0,0.0
4,KazLLM-1.0-8B,LLM-only,NaN,NaN,NaN,NaN,NaN,1.0,1.0
5,Qwen2.5-7B,LLM-only,NaN,NaN,NaN,NaN,NaN,0.0,0.0


Saved: table22_generator_backbone_comparison.csv, table22_generator_backbone_comparison.xlsx
Judge = Claude (non-GPT). Self-preference bias avoided. Pin model snapshot + date in paper.
